<a href="https://colab.research.google.com/github/gbouras13/plassembler/blob/main/run_plassembler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Plassembler

[plassembler](https://github.com/gbouras13/plassembler) is a program that is designed for automated & fast assembly of plasmids in  bacterial genomes that have been hybrid sequenced with long read & paired-end short read sequencing. It was originally designed for Oxford Nanopore Technologies long reads, but it will also work with Pacbio reads. As of v1.3.0, it also works well for long-read only assembled genomes.

The full documentation for Plassembler can be found [here](https://plassembler.readthedocs.io/en/latest).

**To run the code cells, press the play buttons on the top left of each block**

Main Instructions

* Cells 1 and 2 installs plassembler and download the databases. These must be run first.
* Once they have been run, you can run either Cell 3, Cell 4 or both as many times as you wish.
* To run `plassembler run` (if you have both long- and short-reads), run Cell 3.
* To run `plassembler long` (if you have only long-reads), run Cell 4.

Other instructions

* Please make sure you change the runtime to CPU (GPU is not required).
* To do this, go to the top toolbar, then to Runtime -> Change runtime type -> Hardware accelerator
* You may want to upload your FASTQ files first as this takes a while for large files
* Click on the folder icon to the left and use file upload button (with the upwards facing arrow)


In [12]:

#@title 1. Install miniforge and plassembler

#@markdown This cell installs miniforge and plassembler

%%bash

set -e

PYTHON_VERSION=$(python3 -c "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')")
PLASSEMBLER_VERSION="1.8.1"
echo "python version ${PYTHON_VERSION}"

if [ ! -f CONDA_READY ]; then
  echo "installing miniforge"

  # miniforge 25.9.1 introduces some issue - latest as of 7 Nov 2025 - see https://github.com/gbouras13/phold/issues/106
  # issue is fixed if you use the previous release (25.3.1)
  #wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
  wget -qnc https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-25.3.1-0-Linux-x86_64.sh
  bash Miniforge3-25.3.1-0-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
  conda config --set auto_update_conda false
  touch CONDA_READY
fi

if [ ! -f PLASSEMBLER_READY ]; then
  conda install -c bioconda plassembler==$PLASSEMBLER_VERSION
  pip install click
  touch PLASSEMBLER_READY
fi



python version 3.12
Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - plassembler==1.8.1


The following packages will be UPDATED:

  ca-certificates                      2025.7.14-hbd8a1cb_0 --> 2026.1.4-hbd8a1cb_0 


Proceed ([y]/n)? 

Preparing transaction: - done
Verifying transaction: | done
Executing transaction: - done


In [15]:
#@title 2. Download plassembler database

#@markdown This cell downloads the plassembler database.
#@markdown It will take some time (5-10 mins). Please be patient.

%%bash
echo "Downloading plassembler database. This will take some time. Please be patient :)"

plassembler download -d plassembler_db


|████████████████████████████████████████| 381.1M/381.1M [100%] in 3:18.1 (1.92M/s) 


2026-01-24 05:53:52.418 | INFO     | plassembler:download:1185 - Checking database installation at plassembler_db
2026-01-24 05:53:52.418 | INFO     | plassembler.utils.db:check_db_installation:60 - Database directory is missing file plassembler_db/plsdb_2023_11_03_v2.msh. Plassembler Database will be downloaded.
2026-01-24 05:53:52.419 | INFO     | plassembler.utils.db:get_database_zenodo:72 - Downloading Plassembler Database to the directory plassembler_db
2026-01-24 05:57:12.064 | INFO     | plassembler.utils.db:get_database_zenodo:95 - Database file download OK: 3a24bacc05bb857dc044fc6662b58db7
2026-01-24 05:57:12.064 | INFO     | plassembler.utils.db:get_database_zenodo:101 - Extracting DB tarball: file=plassembler_db/201123_plassembler_v1.5.0_databases.tar.gz, output=plassembler_db
2026-01-24 05:57:17.352 | INFO     | plassembler.utils.db:get_database_zenodo:104 - Plassembler Database download into plassembler_db successful.


In [16]:
#@title 3. Plassembler Run (Hybrid reads)

#@markdown This will probably take a while (depends on your read sets - an hour or two probably: best to put it on over lunch) as the colab environment has limited resources.

#@markdown First, upload your long-reads as a single input .fastq or .fastq.gz file

#@markdown Click on the folder icon to the left and use file upload button.

#@markdown Once it is uploaded, write the file name in the LONG_FASTQ field on the right.

#@markdown Then, upload your short-reads as 2 input .fastq or .fastq.gz files

#@markdown Click on the folder icon to the left and use file upload button.

#@markdown Once they are uploaded, write the forward (R1) file name in the R1_FASTQ field on the right and the the reverse (R2) file name in the R2_FASTQ field on the right.

#@markdown Then provide a directory for plassembler's output using PLASSEMBLER_OUT_DIR.
#@markdown The default is 'plassembler_output'.

#@markdown Then provide an estimated chromosome size (as an integer) name using CHROMOSOME.
#@markdown The default is 1000000.

#@markdown You can also provide a min_length for QC filtering the long read data with MIN_LENGTH.
#@markdown If you provide nothing it will default to 1000.

#@markdown You can also provide a min_quality for QC filtering the long read data with MIN_QUALITY.
#@markdown If you provide nothing it will default to 9.

#@markdown You can click SKIP_QC to turn off qc (fastp and filtlong).
#@markdown By default it is False.

#@markdown You can click RAW to  pass --nano-raw for Flye.  Designed for Guppy fast
#@markdown configuration reads.  By default, Flye will assume
#@markdown SUP or HAC reads and use --nano-hq.

#@markdown If you have pacbio reads, please change PACBIO_MODEL
#@markdown from 'none' to one of 'pacbio-hifi', 'pacbio-corr', 'pacbio-raw'. Use pacbio-raw for
#@markdown  PacBio regular CLR reads (<20 percent error),
#@markdown pacbio-corr for PacBio reads that were corrected with other methods (<3 percent error) or pacbio-
#@markdown  hifi for PacBio HiFi reads (<1 percent error).

#@markdown You can click FORCE to overwrite the output directory.
#@markdown This may be useful if your earlier pharokka run has crashed for whatever reason.

#@markdown The results of `plassembler run` will be in the folder icon on the left hand panel.
#@markdown Additionally, it will be zipped so you can download the whole directory.

#@markdown The file to download is PLASSEMBLER_OUT_DIR.zip, where PLASSEMBLER_OUT_DIR is what you provided

#@markdown If you do not see the output directory,
#@markdown refresh the window by either clicking the folder with the refresh icon below "Files"
#@markdown or double click and select "Refresh".
python
import os
import sys
import subprocess
import zipfile

LONG_FASTQ = 'input_fastq.gz' #@param {type:"string"}
R1_FASTQ = 'input_R1.fastq.gz' #@param {type:"string"}
R2_FASTQ = 'input_R2.fastq.gz' #@param {type:"string"}
THREADS = "2"

if os.path.exists(LONG_FASTQ):
    print(f"Input file {LONG_FASTQ} exists")
else:
    print(f"Error: File {LONG_FASTQ} does not exist")
    print(f"Please check the spelling and that you have uploaded it correctly")
    sys.exit(1)

if os.path.exists(R1_FASTQ):
    print(f"Input file {R1_FASTQ} exists")
else:
    print(f"Error: File {R1_FASTQ} does not exist")
    print(f"Please check the spelling and that you have uploaded it correctly")
    sys.exit(1)

if os.path.exists(R2_FASTQ):
    print(f"Input file {R2_FASTQ} exists")
else:
    print(f"Error: File {R2_FASTQ} does not exist")
    print(f"Please check the spelling and that you have uploaded it correctly")
    sys.exit(1)

PLASSEMBLER_OUT_DIR = 'plassembler_output'  #@param {type:"string"}
CHROMOSOME = 50000  #@param {type:"integer"}
MIN_LENGTH = 1000  #@param {type:"integer"}
MIN_QUALITY = 9  #@param {type:"integer"}
SKIP_QC = False  #@param {type:"boolean"}
RAW = False  #@param {type:"boolean"}
PACBIO_MODEL = 'none'  #@param {type:"string"}
allowed_gene_predictors = ['none', 'pacbio-hifi', 'pacbio-corr', 'pacbio-raw']
# Check if the input parameter is valid
if PACBIO_MODEL.lower() not in allowed_gene_predictors:
    raise ValueError("Invalid PACBIO_MODEL. Please choose from: 'none', 'pacbio-hifi', 'pacbio-corr', 'pacbio-raw'.")

FORCE = True  #@param {type:"boolean"}


# Construct the command
command = f"plassembler run -d plassembler_db -c {CHROMOSOME} -l {LONG_FASTQ} -1 {R1_FASTQ} -2 {R2_FASTQ} -o {PLASSEMBLER_OUT_DIR} -t {THREADS} --min_length {MIN_LENGTH} --min_quality {MIN_QUALITY}"

if SKIP_QC is True:
  command = f"{command} --skip_qc"

if RAW is True:
  command = f"{command} -r"

if FORCE is True:
  command = f"{command} -f"

if PACBIO_MODEL != 'none':
  command = f"{command}  --pacbio_model {PACBIO_MODEL}"


# Execute the command
try:
    print("Running plassembler run")
    subprocess.run(command, shell=True, check=True)
    print("plassembler run completed successfully.")
    print(f"Your output is in {PLASSEMBLER_OUT_DIR}.")
    print(f"Zipping the output directory so you can download it all in one go.")

    zip_filename = f"{PLASSEMBLER_OUT_DIR}.zip"

    # Zip the contents of the output directory
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(PLASSEMBLER_OUT_DIR):
            for file in files:
                zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), PLASSEMBLER_OUT_DIR))
    print(f"Output directory has been zipped to {zip_filename}")


except subprocess.CalledProcessError as e:
    print(f"Error occurred: {e}")





Input file input_fastq.gz exists
Input file input_R1.fastq.gz exists
Input file input_R2.fastq.gz exists
Running plassembler run
plassembler run completed successfully.
Your output is in plassembler_output.
Zipping the output directory so you can download it all in one go.
Output directory has been zipped to plassembler_output.zip


/usr/local/lib/python3.12/site-packages/click/core.py:1223: UserWarning: The parameter --flye_directory is used more than once. Remove its duplicate as parameters should be unique.
  parser = self.make_parser(ctx)
/usr/local/lib/python3.12/site-packages/click/core.py:1216: UserWarning: The parameter --flye_directory is used more than once. Remove its duplicate as parameters should be unique.
  self.parse_args(ctx, args)
2026-01-24 06:10:36.159 | INFO     | plassembler:begin_plassembler:84 - --force was specified even though the directory plassembler_output does not already exist. Continuing 
2026-01-24 06:10:36.167 | INFO     | plassembler:begin_plassembler:101 - You are using Plassembler version 1.8.1

2026-01-24 06:10:36.167 | INFO     | plassembler:begin_plassembler:102 - Repository homepage is https://github.com/gbouras13/plassembler
2026-01-24 06:10:36.167 | INFO     | plassembler:begin_plassembler:103 - Written by George Bouras: george.bouras@adelaide.edu.au
2026-01-24 06:10:36.1

In [17]:
#@title 4. Plassembler Long (long-reads only)

#@markdown This will probably take a while (best to put it on overnight or over lunch) as the colab environment has limited resources.

#@markdown First, upload your long-reads as a single input .fastq or .fastq.gz file

#@markdown Click on the folder icon to the left and use file upload button.

#@markdown Once it is uploaded, write the file name in the LONG_FASTQ field on the right.

#@markdown Then provide a directory for plassembler's output using PLASSEMBLER_OUT_DIR.
#@markdown The default is 'plassembler_output'.

#@markdown Then provide an estimated chromosome size (as an integer) name using CHROMOSOME.
#@markdown The default is 1000000.

#@markdown You can also provide a min_length for QC filtering the long read data with MIN_LENGTH.
#@markdown If you provide nothing it will default to 1000.

#@markdown You can also provide a min_quality for QC filtering the long read data with MIN_QUALITY.
#@markdown If you provide nothing it will default to 9.

#@markdown You can click SKIP_QC to turn off qc (filtlong).
#@markdown By default it is False.

#@markdown You can click RAW to  pass --nano-raw for Flye.  Designed for Guppy fast
#@markdown configuration reads.  By default, Flye will assume
#@markdown SUP or HAC reads and use --nano-hq.

#@markdown If you have pacbio reads, please change PACBIO_MODEL
#@markdown from 'none' to one of 'pacbio-hifi', 'pacbio-corr', 'pacbio-raw'. Use pacbio-raw for
#@markdown  PacBio regular CLR reads (<20 percent error),
#@markdown pacbio-corr for PacBio reads that were corrected with other methods (<3 percent error) or pacbio-
#@markdown  hifi for PacBio HiFi reads (<1 percent error).

#@markdown You can click FORCE to overwrite the output directory.
#@markdown This may be useful if your earlier pharokka run has crashed for whatever reason.

#@markdown The results of `plassembler long` will be in the folder icon on the left hand panel.
#@markdown Additionally, it will be zipped so you can download the whole directory.

#@markdown The file to download is PLASSEMBLER_OUT_DIR.zip, where PLASSEMBLER_OUT_DIR is what you provided

#@markdown If you do not see the output directory,
#@markdown refresh the window by either clicking the folder with the refresh icon below "Files"
#@markdown or double click and select "Refresh".

python
import os
import sys
import subprocess
import zipfile
LONG_FASTQ = 'input_fastq.gz' #@param {type:"string"}
THREADS = "2"

if os.path.exists(LONG_FASTQ):
    print(f"Input file {LONG_FASTQ} exists")
else:
    print(f"Error: File {LONG_FASTQ} does not exist")
    print(f"Please check the spelling and that you have uploaded it correctly")
    sys.exit(1)


PLASSEMBLER_OUT_DIR = 'plassembler_output'  #@param {type:"string"}
CHROMOSOME = 50000  #@param {type:"integer"}
MIN_LENGTH = 1000  #@param {type:"integer"}
MIN_QUALITY = 9  #@param {type:"integer"}
SKIP_QC = False  #@param {type:"boolean"}
RAW = False  #@param {type:"boolean"}
PACBIO_MODEL = 'none'  #@param {type:"string"}
allowed_gene_predictors = ['none', 'pacbio-hifi', 'pacbio-corr', 'pacbio-raw']
# Check if the input parameter is valid
if PACBIO_MODEL.lower() not in allowed_gene_predictors:
    raise ValueError("Invalid PACBIO_MODEL. Please choose from: 'none', 'pacbio-hifi', 'pacbio-corr', 'pacbio-raw'.")

FORCE = True  #@param {type:"boolean"}

# Construct the command
command = f"plassembler long -d plassembler_db -c {CHROMOSOME} -l {LONG_FASTQ} -o {PLASSEMBLER_OUT_DIR} -t {THREADS} --min_length {MIN_LENGTH} --min_quality {MIN_QUALITY}"

if SKIP_QC is True:
  command = f"{command} --skip_qc"

if RAW is True:
  command = f"{command} -r"

if FORCE is True:
  command = f"{command} -f"

if PACBIO_MODEL != 'none':
  command = f"{command}  --pacbio_model {PACBIO_MODEL}"

# Execute the command
try:
    print("Running plassembler long")
    subprocess.run(command, shell=True, check=True)
    print("plassembler long completed successfully.")
    print(f"Your output is in {PLASSEMBLER_OUT_DIR}.")
    print(f"Zipping the output directory so you can download it all in one go.")

    zip_filename = f"{PLASSEMBLER_OUT_DIR}.zip"

    # Zip the contents of the output directory
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(PLASSEMBLER_OUT_DIR):
            for file in files:
                zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), PLASSEMBLER_OUT_DIR))
    print(f"Output directory has been zipped to {zip_filename}")


except subprocess.CalledProcessError as e:
    print(f"Error occurred: {e}")



Input file input_fastq.gz exists
Running plassembler long
plassembler long completed successfully.
Your output is in plassembler_output.
Zipping the output directory so you can download it all in one go.
Output directory has been zipped to plassembler_output.zip



EnvironmentLocationNotFound: Not a conda environment: /usr/local/envs/plassemblerENV

2026-01-24 06:19:47.548 | INFO     | plassembler:begin_plassembler:101 - You are using Plassembler version 1.8.1

2026-01-24 06:19:47.549 | INFO     | plassembler:begin_plassembler:102 - Repository homepage is https://github.com/gbouras13/plassembler
2026-01-24 06:19:47.549 | INFO     | plassembler:begin_plassembler:103 - Written by George Bouras: george.bouras@adelaide.edu.au
2026-01-24 06:19:47.549 | INFO     | plassembler:long:1371 - Database directory is plassembler_db
2026-01-24 06:19:47.549 | INFO     | plassembler:long:1372 - Longreads file is input_fastq.gz
2026-01-24 06:19:47.549 | INFO     | plassembler:long:1373 - Chromosome length threshold is 50000
2026-01-24 06:19:47.549 | INFO     | plassembler:long:1374 - Output directory is plassembler_output
2026-01-24 06:19:47.549 | INFO     | plassembler:long:1375 - Min long read length is 1000
2026-01-24 06:19:47.549 | INFO     | plassembler:long